## Подключение

Ячейка подключения выполняется первой в каждом сеансе. Она подключает Google Drive (личная папка `practice2026` и общие данные группы `kg_data`), клонирует репозиторий группы в `/content/kg-practice-2026`, переключается на свою ветку `student-NN` с работой группы из `main` и регистрирует модули группы. В первых строках студент указывает свой номер, модуль и фамилию латиницей; токен GitHub лежит в секретах Colab под именем `GITHUB_TOKEN` (`ИНСТРУКЦИЯ_git.md`). Сдача этапа — последняя ячейка блокнота: `repo.submit(...)`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import sys, os, json, time, subprocess

STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами
REPO = 'github.com/<организация>/kg-practice-2026'     # репозиторий группы

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
ONTOLOGY = [KG + '/ontology/MocassinPlus_v2.ttl', KG + '/ontology/SciLibReco.ttl']
os.makedirs(WORK, exist_ok=True)

TOKEN = userdata.get('GITHUB_TOKEN')                   # секрет Colab с токеном GitHub
if not os.path.exists(KG):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{REPO}.git', KG], capture_output=True, text=True)
    print(r.stderr.replace(TOKEN, '***'))
sys.path.insert(0, KG)
!pip install -q -r {KG}/requirements.txt
from mathkg.gitwork import Repo
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print('свой модуль:', MY_MODULE, my_tool)

# Этап 02, приложение. Разбор base по шагам

Блокнот к этапу 2. Функция `load_doc` превращает PDF статьи в `Doc` за один вызов; здесь тот же путь
пройден по шагам на одной статье общего тестового набора. На каждом шаге видно, что делает
внутренняя функция `base`, какие данные она получает и отдаёт и где возникают ошибки, которые
студент записывает в задачу (issue) репозитория группы с меткой `base`. Код `base` при этом не копируется: блокнот
вызывает функции пакета `mathkg.base`, исходник лежит в `mathkg/base/__init__.py` в репозитории группы.

Время: 1–1,5 ч. GPU не нужен.

## Шаг 0. Статья для разбора

Студент выбирает статью тестового набора. Для первого прохода удобна статья, свёрстанная LaTeX
(в ней есть курсив, математические шрифты и гиперссылки); для второго — статья из Word на русском
языке.

In [ ]:
import csv, re, json
from collections import Counter
import fitz                                         # PyMuPDF
from PIL import Image, ImageDraw
import mathkg.base as B

rows = list(csv.DictReader(open(TEST + '/index.csv', encoding='utf-8')))
for r in rows[:30]:
    print(r['doc_id'], r['file'], r.get('source', ''))
DOC_ID = rows[0]['doc_id']                           # ← номер статьи для разбора
PDF = f"{TEST}/pdf/{next(r['file'] for r in rows if r['doc_id'] == DOC_ID)}"
pdf = fitz.open(PDF)
print(DOC_ID, 'страниц:', len(pdf), '| метаданные PDF:', {k: v for k, v in pdf.metadata.items() if v})


def show(img, width=900):
    """Показывает картинку в блокноте (в Colab) или сохраняет её в файл (вне блокнота)."""
    img = img.copy(); img.thumbnail((width, 10 * width))
    try:
        from IPython.display import display
        display(img)
    except ImportError:
        img.save(f'{WORK}/_show.png')


def page_image(page, zoom=1.5):
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    return Image.frombytes('RGB', (pix.width, pix.height), pix.samples)

## Шаг 1. Строки PDF со шрифтами — `read_pdf`

PDF хранит символы и их координаты, логической структуры в нём нет. `read_pdf` читает каждую
страницу через `page.get_text("dict")`: блоки → строки → span-ы (отрезки одного шрифта). Для каждой
строки base запоминает текст, рамку `bbox`, основной кегль, признаки «вся строка полужирная / курсив»
и долю символов математического шрифта `math_share`.

In [ ]:
pages, links_raw, meta, widths = B.read_pdf(PDF)
print('строк по страницам:', [len(p) for p in pages])
for l in pages[0][:12]:
    print(f"{l['size']:5.1f} b={int(l['bold'])} i={int(l['italic'])} math={l['math_share']:.2f} "
          f"x0={l['bbox'][0]:6.1f} y0={l['bbox'][1]:6.1f} | {l['text'][:70]}")

Одна строка состоит из нескольких span-ов: по ним видно, где кончается полужирный заголовок блока и
начинается текст утверждения.

In [ ]:
line = next((l for p in pages for l in p if len(l['spans']) >= 3), pages[0][0])
for s in line['spans']:
    print(f"{s['font']:28} {s['size']:5.1f} bold={s['bold']} italic={s['italic']} math={s['math']} | {s['text']!r}")

fonts = Counter()
for p in pages:
    for l in p:
        for s in l['spans']:
            fonts[(s['font'], s['size'])] += len(s['text'])
print('самые частые шрифты (по числу символов):')
for (f, sz), n in fonts.most_common(8):
    print(f'  {f:30} {sz:5.1f}  {n}')

Самый частый кегль — основной текст статьи. От него base отсчитывает сноски (мельче основного) и
заголовок статьи (крупнее).

## Шаг 2. Порядок чтения — `_column_order`

`get_text` отдаёт блоки в порядке, в котором они записаны в файле, и в двухколоночной вёрстке этот
порядок часто не совпадает с порядком чтения. base делит страницу на полосы полноширинными блоками
и внутри полосы читает сначала левую колонку, затем правую. На картинке у каждого блока — номер в
порядке чтения base.

In [ ]:
PAGE = 1                                             # ← страница (с 1)
page = pdf[PAGE - 1]
blocks = [b for b in page.get_text('dict')['blocks'] if b.get('type', 0) == 0]
ordered = B._column_order(blocks, page.rect.width)
img = page_image(page); d = ImageDraw.Draw(img); z = 1.5
for k, b in enumerate(ordered, 1):
    x0, y0, x1, y1 = [v * z for v in b['bbox']]
    d.rectangle([x0, y0, x1, y1], outline=(31, 122, 115), width=3)
    d.rectangle([x0, y0, x0 + 34, y0 + 22], fill=(217, 142, 4))
    d.text((x0 + 4, y0 + 4), str(k), fill='white')
show(img)

## Шаг 3. Колонтитулы и номера страниц — `find_running_lines`

Колонтитулы повторяются на многих страницах и разрывают текст. base берёт две верхние и две нижние
строки каждой страницы, заменяет в них цифры на `#` и считает, на скольких страницах встречается
каждая такая «форма». Формы, которые встречаются хотя бы на трети страниц, считаются колонтитулами.

In [ ]:
running = B.find_running_lines(pages)
print('формы колонтитулов:', running)
for pno, lines in enumerate(pages[:4], start=1):
    by_y = sorted(lines, key=lambda l: l['bbox'][1])
    for l in by_y[:2] + by_y[-2:]:
        mark = 'УДАЛЯЕТСЯ' if B._shape(l['text']) in running else 'остаётся '
        print(pno, mark, '|', l['text'][:80])

## Шаг 4. Символы — `norm_char`, `garbage_share`, `check_text_layer`

Текстовый слой PDF содержит символы, которые мешают поиску: лигатуры (ﬁ, ﬂ), математические буквы
Unicode (𝑥, 𝜑), символы шрифта Symbol из PDF, свёрстанных в Word (частная зона U+F020–U+F0FF),
управляющие символы. `norm_char` приводит их к обычным символам, а `garbage_share` считает долю
символов, характерных для сломанной кодировки шрифта.

In [ ]:
for ch in ['ﬁ', 'ﬂ', '𝑥', '𝜑', '𝔽', '', '', '', '', '\x10', '²', 'ё']:
    print(f'{ch!r:10} U+{ord(ch):05X} -> {B.norm_char(ch)!r}')

all_text = ' '.join(l['text'] for p in pages for l in p)
print('доля «мусорных» символов:', round(B.garbage_share(all_text), 4))
print('символов на страницу:', len(all_text) // max(1, len(pages)))

Знак «²» и букву «ё» base не меняет: полная нормализация NFKC превратила бы верхний индекс в обычную
цифру, и запись x² стала бы x2. Поэтому NFKC применяется только к диапазону математических букв.

## Шаг 5. Выключные формулы — `is_display_formula`, `NUM_RIGHT`, `NUM_ONLY`

Выключная формула — отдельная строка, в основном из математических символов, с отступом от края
текста или с номером в скобках у правого края. Номер бывает частью строки («… = 0, (2.3)») или
отдельной строкой справа («(2.3)»).

In [ ]:
sizes = Counter()
for p in pages:
    for l in p:
        sizes[l['size']] += len(l['text'])
body = sizes.most_common(1)[0][0]
lefts = Counter(round(l['bbox'][0]) for p in pages for l in p if abs(l['size'] - body) < 0.6)
rights = Counter(round(l['bbox'][2]) for p in pages for l in p if abs(l['size'] - body) < 0.6)
body_left = lefts.most_common(1)[0][0]
body_right = max(r for r, c in rights.items() if c >= 5)
print('основной кегль', body, '| левый край текста', body_left, '| правый', body_right)

cands = []
for pno, lines in enumerate(pages, start=1):
    for l in lines:
        t = l['text'].strip()
        if B.is_display_formula(l, body_left, body_right) or B.NUM_ONLY.match(t):
            m = B.NUM_RIGHT.search(t) or B.NUM_ONLY.match(t)
            cands.append((pno, l, m.group(1) if m else None))
print('строк-кандидатов:', len(cands))
for pno, l, num in cands[:12]:
    print(pno, f"math={l['math_share']:.2f}", 'номер', num, '|', l['text'][:70])

Вырезка формулы из изображения страницы показывает, что теряется в текстовом слое: индексы,
дроби, знаки сумм. Именно такую вырезку base передаёт модели pix2tex в `recognize_formulas`.

In [ ]:
for pno, l, num in [c for c in cands if not B.NUM_ONLY.match(c[1]['text'].strip())][:3]:
    x0, y0, x1, y1 = l['bbox']
    clip = fitz.Rect(x0 - 3, y0 - 3, x1 + 3, y1 + 3)
    pix = pdf[pno - 1].get_pixmap(matrix=fitz.Matrix(3, 3), clip=clip)
    show(Image.frombytes('RGB', (pix.width, pix.height), pix.samples), width=600)
    print('текст из PDF:', l['text'])

## Шаг 6. Сборка `Doc` — `assemble`

`assemble` собирает очищенный текст: убирает колонтитулы и сноски, склеивает переносы
(«простран-» + «ство»), начинает новый абзац при смене блока, вертикальном зазоре, красной строке
или заголовке блока, выделяет выключные формулы и переносит отметки шрифта и гиперссылки в позиции
текста. Результат совпадает с тем, что возвращает `load_doc`.

In [ ]:
doc = B.assemble(DOC_ID, pages, links_raw, widths, meta)
print(doc.lang, 'символов', len(doc.text), '| абзацев', len(doc.paragraphs), '| предложений', len(doc.sentences),
      '| формул', len(doc.formulas), '(с номером', sum(1 for f in doc.formulas if f.number), ') | сносок', len(doc.footnotes))
same = B.load_doc(PDF, DOC_ID)
print('совпадает с load_doc:', same.text == doc.text and len(same.paragraphs) == len(doc.paragraphs))

Переносы: строка PDF, которая кончается дефисом перед строчной буквой, склеивается со следующей без
дефиса. Ниже — несколько таких мест и результат в тексте `Doc`.

In [ ]:
shown = 0
for lines in pages:
    for a, b in zip(lines, lines[1:]):
        if re.search(r'[A-Za-zА-Яа-яЁё]-$', a['text'].rstrip()) and re.match(r'\s*[a-zа-яё]', b['text']):
            left = re.findall(r'(\w+)-$', a['text'].rstrip())[0]
            right = re.match(r'\s*(\w+)', b['text']).group(1)
            print(f'{left}- + {right}  ->  в тексте Doc:', (left + right) in doc.text)
            shown += 1
            if shown >= 6:
                break

Абзацы и формулы по порядку на одной странице:

In [ ]:
items = [(p.start, 'абзац  ', doc.text[p.start:p.end]) for p in doc.paragraphs if p.page == PAGE]
items += [(f.start, f'формула ({f.number})' if f.number else 'формула', f.text) for f in doc.formulas if f.page == PAGE]
for start, kind, t in sorted(items)[:20]:
    print(f'{start:6} {kind:14} | {t[:90]}')
print('сноски:', doc.footnotes[:3])

## Шаг 7. Отметки шрифта — `doc.styled`

Шрифт — разметка, которую автор статьи уже сделал: заголовок блока «Теорема 2.1.» полужирный,
формулировка теоремы и вводимый термин — курсивом, формулы — математическим шрифтом. base переносит
эти отметки в позиции очищенного текста (`Span(start, end, page, kind)`), и модули 3–5 ищут по ним
границы утверждений, термины определений и строчные формулы.

In [ ]:
for kind in ('bold', 'italic', 'math'):
    sp = [s for s in doc.styled if s.kind == kind]
    print(f'{kind:7} {len(sp):5} отрезков | примеры:', [doc.text[s.start:s.end][:30] for s in sp[:5]])

## Шаг 8. Язык и предложения — `detect_lang`, razdel, pysbd

Язык определяется по доле кириллицы. Русский текст делится на предложения библиотекой razdel,
английский — pysbd. Сокращения научного текста («т. е.», «см.», «Thm.», «et al.») и номера
(«Lemma 2.1.») — частые места ошибок.

In [ ]:
print('язык:', B.detect_lang(doc.text[:20000]))
par = max(doc.paragraphs, key=lambda p: p.end - p.start)
for s in [s for s in doc.sentences if par.start <= s.start < par.end][:8]:
    print('·', doc.text[s.start:s.end][:110])

## Шаг 9. Гиперссылки PDF — `doc.links`

В статьях, свёрстанных LaTeX с пакетом hyperref, ссылки «[12]», «(2.3)», «Theorem 1.2» — это
гиперссылки с именованной целью (`cite.…`, `equation.2.3`, `theorem.1.2`). base находит текст ссылки
по рамке на странице и записывает его позицию в `Doc.text`; модуль 8 подтверждает этими ссылками
найденные связи.

In [ ]:
print('ссылок в PDF:', len(links_raw), '| сопоставлено с текстом:', sum(1 for l in doc.links if l.start >= 0))
print('виды целей:', Counter(l.target.split('.')[0] for l in doc.links).most_common(8))
for l in doc.links[:10]:
    print(f'{l.target:40} стр. {l.page} -> {l.target_page} |', repr(doc.text[l.start:l.end]) if l.start >= 0 else '—')

У статей из Word гиперссылок с целями обычно нет, и этот шаг для них пуст.

## Упражнения

1. **Колонтитулы.** Найти в тестовом наборе статью, где колонтитул остался в тексте (`find_running_lines`
   не нашёл его форму), объяснить причину (меняется не только номер страницы, колонтитул стоит не в
   двух крайних строках, страниц меньше трёх) и записать случай в задачу (issue) репозитория группы с меткой `base`.
2. **Формулы.** На одной странице сравнить список выключных формул base с картинкой страницы:
   сколько формул пропущено, сколько лишних (строки таблиц, кода), сколько формул разорвано на части.
   Числа — в отчёт этапа.
3. **Порядок чтения.** Найти двухколоночную статью и проверить на картинке шага 2, что номера блоков
   идут в порядке чтения. Если нет — записать случай в `base_issues.md` с номером страницы.
4. **Свой модуль.** Для своего модуля назвать признаки base, на которые опираются его правила (шрифт,
   положение строки, номера формул, гиперссылки), и проверить на трёх статьях, что эти признаки в
   `Doc` есть. Вывод — один абзац в отчёте этапа 2.

Запись в `base_issues.md`: дата, `doc_id`, страница, что не так, фрагмент текста. Сам код base
студенты не правят; руководитель исправляет его централизованно, и исправление получают все.